In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime
from scipy import stats
from scipy.stats import f_oneway
from tqdm import tqdm
import warnings
import gc
from concurrent.futures import ThreadPoolExecutor, as_completed

warnings.filterwarnings('ignore')

# ============================================================================
# CONFIGURATION SECTION - USER MUST PROVIDE PATHS HERE
# ============================================================================

# Define paths for each plate's data
# Format: {plate_id: {'parquet': path, 'platemap': path}}
PLATE_PATHS = {
    'P1': {
        'parquet': r"D:\2025_12_19 CRISPRi Reference Plate Imaging\P1\CellposeSAM Segmentation results\micromorph_cell_measurements.parquet",
        'platemap': r"D:\2025_12_19 CRISPRi Reference Plate Imaging\P1\CellposeSAM Segmentation results\plate_map.xlsx"
    },
    'P2': {
        'parquet': r"D:\2025_12_19 CRISPRi Reference Plate Imaging\P2\CellposeSAM Segmentation results\micromorph_cell_measurements.parquet",
        'platemap': r"D:\2025_12_19 CRISPRi Reference Plate Imaging\P2\CellposeSAM Segmentation results\plate_map.xlsx"
    },
    'P3': {
        'parquet': r"D:\2025_12_19 CRISPRi Reference Plate Imaging\P3\CellposeSAM Segmentation results\micromorph_cell_measurements.parquet",
        'platemap': r"D:\2025_12_19 CRISPRi Reference Plate Imaging\P3\CellposeSAM Segmentation results\plate_map.xlsx"
    },
    'P4': {
        'parquet': r"D:\2025_12_19 CRISPRi Reference Plate Imaging\P4\CellposeSAM Segmentation results\micromorph_cell_measurements.parquet",
        'platemap': r"D:\2025_12_19 CRISPRi Reference Plate Imaging\P4\CellposeSAM Segmentation results\plate_map.xlsx"
    },
    'P5': {
        'parquet': r"D:\2025_12_19 CRISPRi Reference Plate Imaging\P5\CellposeSAM Segmentation results\micromorph_cell_measurements.parquet",
        'platemap': r"D:\2025_12_19 CRISPRi Reference Plate Imaging\P5\CellposeSAM Segmentation results\plate_map.xlsx"
    },
    'P6': {
        'parquet': r"D:\2025_12_19 CRISPRi Reference Plate Imaging\P6\CellposeSAM Segmentation results\micromorph_cell_measurements.parquet",
        'platemap': r"D:\2025_12_19 CRISPRi Reference Plate Imaging\P6\CellposeSAM Segmentation results\plate_map.xlsx"
    }
}

# Output directory
OUTPUT_BASE_FOLDER = r"D:\2025_12_19 CRISPRi Reference Plate Imaging\multiplate_analysis"

# Analysis parameters (matching single-plate script)
MORPHOLOGY_FEATURES = ['roundness', 'area_ÂµmÂ²', 'length_Âµm', 'width_Âµm', 'perimeter_Âµm']
BIN_WIDTHS = {'roundness': 0.02, 'area_ÂµmÂ²': 0.2, 'length_Âµm': 0.05, 'width_Âµm': 0.02, 'perimeter_Âµm': 0.2}
FEATURE_UNITS = {'roundness': '', 'area_ÂµmÂ²': 'ÂµmÂ²', 'length_Âµm': 'Âµm', 'width_Âµm': 'Âµm', 'perimeter_Âµm': 'Âµm'}
HISTOGRAM_ALPHA = 0.8
FIGURE_SIZE = (12, 6)
DPI = 150
EFFECT_SIZE_THRESHOLDS = {'small': 0.2, 'medium': 0.5, 'large': 0.8}

# ============================================================================
# UTILITIES (from single-plate script)
# ============================================================================

class EffectSizeCalculator:
    @staticmethod
    def cohens_d(group1, group2):
        """Optimized Cohen's d calculation using float32 arrays"""
        g1 = np.asarray(group1, dtype=np.float32)
        g2 = np.asarray(group2, dtype=np.float32)
        n1, n2 = len(g1), len(g2)
        
        if n1 < 2 or n2 < 2:
            return np.nan
        
        mean1, mean2 = np.mean(g1), np.mean(g2)
        var1, var2 = np.var(g1, ddof=1), np.var(g2, ddof=1)
        pooled_sd = np.sqrt(((n1 - 1) * var1 + (n2 - 1) * var2) / (n1 + n2 - 2))
        
        return float((mean1 - mean2) / pooled_sd) if pooled_sd != 0 else np.nan
    
    @staticmethod
    def interpret_cohens_d(d):
        if np.isnan(d):
            return "undefined"
        abs_d = abs(d)
        if abs_d < 0.2:
            return "negligible"
        elif abs_d < 0.5:
            return "small"
        elif abs_d < 0.8:
            return "medium"
        else:
            return "large"


def parse_gene_subgroup(label):
    """Parse gene label into base name and subgroup number"""
    if '_' in label and label != 'WT':
        parts = label.rsplit('_', 1)
        if len(parts) == 2 and parts[1] in ['1', '2', '3']:
            return parts[0], parts[1]
    return label, None


def get_grouped_gene_name(label):
    """Get the grouped gene name (without subgroup suffix)"""
    base_gene, _ = parse_gene_subgroup(label)
    return base_gene


# ============================================================================
# DATA LOADING - MEMORY EFFICIENT
# ============================================================================

def load_single_plate(plate_id, parquet_path, platemap_path):
    """
    Load and prepare data for a single plate.
    Memory efficient: only keeps necessary columns.
    """
    print(f"  Loading plate {plate_id}...")
    
    # Map old column names to new ones
    column_mapping = {
        'area_um2': 'area_ÂµmÂ²',
        'length_um': 'length_Âµm',
        'width_um': 'width_Âµm',
        'perimeter_um': 'perimeter_Âµm'
    }
    
    # Load parquet with old column names
    old_features = ['roundness', 'area_um2', 'length_um', 'width_um', 'perimeter_um']
    df = pd.read_parquet(parquet_path, columns=['Well'] + old_features)
    
    # Rename columns to use Âµm
    df = df.rename(columns=column_mapping)
    
    # Convert features to float32 for memory efficiency
    for feature in MORPHOLOGY_FEATURES:
        if feature in df.columns:
            df[feature] = df[feature].astype(np.float32)
    
    # Rename Well column if needed
    if 'Well' in df.columns:
        df['well'] = df['Well']
        df.drop('Well', axis=1, inplace=True)
    
    # Load plate map
    plate_map = pd.read_excel(platemap_path, header=None)
    
    # Map wells to genes
    def get_gene_label(well):
        if pd.isna(well) or len(str(well)) < 2:
            return None
        try:
            row_idx = ord(well[0].upper()) - ord('A')
            col_idx = int(well[1:]) - 1
            if 0 <= row_idx < plate_map.shape[0] and 0 <= col_idx < plate_map.shape[1]:
                label = plate_map.iloc[row_idx, col_idx]
                return str(label) if pd.notna(label) else None
        except:
            return None
    
    df['gene'] = df['well'].apply(get_gene_label)
    
    # Add plate ID as categorical to save memory
    df['plate'] = pd.Categorical([plate_id] * len(df))
    
    # Filter outliers per feature (1% to 99%)
    initial_count = len(df)
    for feature in MORPHOLOGY_FEATURES:
        if feature in df.columns:
            q1 = df[feature].quantile(0.01)
            q99 = df[feature].quantile(0.99)
            df = df[(df[feature] >= q1) & (df[feature] <= q99)]
    
    # Remove rows with no gene assignment
    df = df.dropna(subset=['gene'])
    
    filtered_count = len(df)
    print(f"  {plate_id}: {filtered_count:,} cells ({initial_count - filtered_count:,} filtered)")
    
    return df


def load_all_plates(max_workers=None):
    """
    Load all plates and combine into single dataframe.
    Memory efficient: loads plates in parallel using multithreading.
    """
    print("="*80)
    print("LOADING MULTI-PLATE DATA (MULTITHREADED)")
    print("="*80)
    
    n_plates = len(PLATE_PATHS)
    if max_workers is None:
        max_workers = min(6, n_plates)
    print(f"  Using {max_workers} threads for parallel loading")
    print()
    
    plate_dfs = []
    
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        future_to_plate = {
            executor.submit(
                load_single_plate,
                plate_id,
                paths['parquet'],
                paths['platemap']
            ): plate_id
            for plate_id, paths in PLATE_PATHS.items()
        }
        
        for future in as_completed(future_to_plate):
            plate_id = future_to_plate[future]
            try:
                df = future.result()
                plate_dfs.append(df)
            except Exception as exc:
                print(f"  ERROR: Plate {plate_id} generated an exception: {exc}")
                raise
    
    print("\n  Combining all plates...")
    combined_df = pd.concat(plate_dfs, ignore_index=True)
    
    combined_df['gene'] = pd.Categorical(combined_df['gene'])
    
    del plate_dfs
    gc.collect()
    
    print(f"\n  Total cells across all plates: {len(combined_df):,}")
    print(f"  Unique plates: {combined_df['plate'].nunique()}")
    print(f"  Unique genes: {combined_df['gene'].nunique()}")
    print(f"  WT cells: {len(combined_df[combined_df['gene'] == 'WT']):,}")
    
    return combined_df


# ============================================================================
# ANALYSIS 1: WT VARIABILITY ACROSS PLATES (PRIMARY FOCUS)
# ============================================================================

class WTPlateVariabilityAnalyzer:
    """
    Analyzes WT variability across plates.
    This is the MOST IMPORTANT analysis.
    """
    
    def __init__(self, df, output_base):
        self.df = df
        self.output_base = Path(output_base)
    
    def analyze_all_features(self):
        """Run analysis for all features"""
        print("\n" + "="*80)
        print("ANALYSIS 1: WT VARIABILITY ACROSS PLATES")
        print("="*80)
        
        output_folder = self.output_base / "01_wt_plate_variability"
        output_folder.mkdir(parents=True, exist_ok=True)
        
        for feature in MORPHOLOGY_FEATURES:
            print(f"\n  Processing: {feature}")
            self._analyze_feature(feature, output_folder)
        
        print("\n  [OK] WT plate variability analysis complete")
    
    def _analyze_feature(self, feature, output_folder):
        """Analyze WT variability for a single feature"""
        wt_df = self.df[self.df['gene'] == 'WT'].copy()
        
        well_stats = []
        for (plate, well), group in wt_df.groupby(['plate', 'well']):
            values = group[feature].values
            if len(values) >= 2:
                well_stats.append({
                    'plate': plate,
                    'well': well,
                    'mean': np.mean(values),
                    'sd': np.std(values, ddof=1),
                    'cv': (np.std(values, ddof=1) / np.mean(values) * 100) if np.mean(values) != 0 else np.nan,
                    'n_cells': len(values)
                })
        
        well_stats_df = pd.DataFrame(well_stats)
        well_stats_df.to_csv(output_folder / f"wt_well_stats_{feature}.csv", index=False)
        
        plate_stats = []
        for plate in sorted(wt_df['plate'].unique()):
            plate_data = wt_df[wt_df['plate'] == plate]
            values = plate_data[feature].values
            plate_stats.append({
                'plate': plate,
                'mean': np.mean(values),
                'sd': np.std(values, ddof=1),
                'cv': (np.std(values, ddof=1) / np.mean(values) * 100) if np.mean(values) != 0 else np.nan,
                'median': np.median(values),
                'q25': np.percentile(values, 25),
                'q75': np.percentile(values, 75),
                'iqr': np.percentile(values, 75) - np.percentile(values, 25),
                'n_cells': len(values),
                'n_wells': len(plate_data['well'].unique())
            })
        
        plate_stats_df = pd.DataFrame(plate_stats)
        plate_stats_df.to_csv(output_folder / f"wt_plate_stats_{feature}.csv", index=False)
        
        self._generate_wt_boxplot(well_stats_df, feature, output_folder)
        self._write_wt_summary(plate_stats_df, well_stats_df, feature, output_folder)
    
    def _generate_wt_boxplot(self, well_stats_df, feature, output_folder):
        """Generate boxplot of WT variability across plates"""
        fig, ax = plt.subplots(figsize=(10, 6), dpi=DPI)
        
        plates = sorted(well_stats_df['plate'].unique())
        data_by_plate = [well_stats_df[well_stats_df['plate'] == plate]['mean'].values 
                        for plate in plates]
        
        bp = ax.boxplot(data_by_plate, labels=plates, patch_artist=True, 
                       widths=0.6, showfliers=False)
        
        for patch in bp['boxes']:
            patch.set_facecolor('lightblue')
            patch.set_alpha(0.7)
        for element in ['whiskers', 'caps', 'medians']:
            plt.setp(bp[element], color='black', linewidth=1.5)
        
        for i, plate in enumerate(plates):
            plate_data = well_stats_df[well_stats_df['plate'] == plate]['mean'].values
            x = np.random.normal(i + 1, 0.04, size=len(plate_data))
            ax.scatter(x, plate_data, alpha=0.6, s=30, color='red', zorder=3)
        
        unit = FEATURE_UNITS.get(feature, '')
        ax.set_xlabel('Plate', fontsize=12, fontweight='bold')
        ax.set_ylabel(f'WT Well Mean {feature} {unit}'.strip(), fontsize=12, fontweight='bold')
        ax.set_title(f'WT Variability Across Plates: {feature}', fontsize=14, fontweight='bold')
        ax.grid(True, alpha=0.3, axis='y', linestyle='--')
        
        # Set y-axis limits
        ax.set_ylim(bottom=0)
        if feature == 'roundness':
            ax.set_ylim(0, 1)
        elif feature == 'area_ÂµmÂ²':
            ax.set_ylim(0, 5)
        elif feature == 'length_Âµm':
            ax.set_ylim(0, 5)
        elif feature == 'width_Âµm':
            ax.set_ylim(0, 2)
        elif feature == 'perimeter_Âµm':
            ax.set_ylim(0, 10)
        
        plt.tight_layout()
        plt.savefig(output_folder / f"wt_plate_variability_boxplot_{feature}.png", 
                   dpi=DPI, bbox_inches='tight')
        plt.close()
    
    def _write_wt_summary(self, plate_stats_df, well_stats_df, feature, output_folder):
        """Write comprehensive summary of WT variability"""
        unit = FEATURE_UNITS.get(feature, '')
        
        with open(output_folder / f"wt_variability_summary_{feature}.txt", 'w') as f:
            f.write("="*80 + "\n")
            f.write(f"WT VARIABILITY ACROSS PLATES: {feature}\n")
            f.write("="*80 + "\n\n")
            
            f.write("PLATE-LEVEL SUMMARY:\n")
            f.write("-"*80 + "\n")
            for _, row in plate_stats_df.iterrows():
                f.write(f"\n{row['plate']}:\n")
                f.write(f"  Mean Â± SD: {row['mean']:.3f} Â± {row['sd']:.3f} {unit}\n")
                f.write(f"  CV: {row['cv']:.2f}%\n")
                f.write(f"  Median [IQR]: {row['median']:.3f} [{row['q25']:.3f} - {row['q75']:.3f}]\n")
                f.write(f"  Sample: n={row['n_cells']:,} cells from {int(row['n_wells'])} wells\n")
            
            f.write("\n" + "="*80 + "\n")
            f.write("BETWEEN-PLATE VARIABILITY:\n")
            f.write("-"*80 + "\n")
            plate_means = plate_stats_df['mean'].values
            f.write(f"  Range of plate means: {np.min(plate_means):.3f} - {np.max(plate_means):.3f} {unit}\n")
            f.write(f"  SD of plate means: {np.std(plate_means, ddof=1):.3f} {unit}\n")
            f.write(f"  CV of plate means: {(np.std(plate_means, ddof=1) / np.mean(plate_means) * 100):.2f}%\n")
            
            f.write("\n" + "="*80 + "\n")
            f.write("WITHIN-PLATE VARIABILITY:\n")
            f.write("-"*80 + "\n")
            well_cvs = well_stats_df.groupby('plate')['cv'].mean()
            f.write(f"  Average within-plate CV: {well_cvs.mean():.2f}%\n")
            for plate, cv in well_cvs.items():
                f.write(f"    {plate}: {cv:.2f}%\n")


# ============================================================================
# ANALYSIS 2: GENE VARIABILITY ACROSS PLATES
# ============================================================================

class GenePlateVariabilityAnalyzer:
    """Analyzes gene phenotype consistency across plates."""
    
    def __init__(self, df, output_base):
        self.df = df
        self.output_base = Path(output_base)
    
    def analyze_all_features(self):
        """Run analysis for all features"""
        print("\n" + "="*80)
        print("ANALYSIS 2: GENE VARIABILITY ACROSS PLATES")
        print("="*80)
        
        output_folder = self.output_base / "02_gene_plate_variability"
        output_folder.mkdir(parents=True, exist_ok=True)
        
        for feature in MORPHOLOGY_FEATURES:
            print(f"\n  Processing: {feature}")
            self._analyze_feature(feature, output_folder)
        
        print("\n  [OK] Gene plate variability analysis complete")
    
    def _analyze_feature(self, feature, output_folder):
        """Analyze gene variability across plates for a single feature"""
        genes = [g for g in self.df['gene'].unique() if g != 'WT']
        
        gene_plate_stats = []
        for gene in genes:
            gene_df = self.df[self.df['gene'] == gene]
            
            for plate in sorted(gene_df['plate'].unique()):
                plate_data = gene_df[gene_df['plate'] == plate]
                values = plate_data[feature].values
                
                if len(values) >= 10:
                    gene_plate_stats.append({
                        'gene': gene,
                        'plate': plate,
                        'mean': np.mean(values),
                        'sd': np.std(values, ddof=1),
                        'cv': (np.std(values, ddof=1) / np.mean(values) * 100) if np.mean(values) != 0 else np.nan,
                        'n_cells': len(values)
                    })
        
        gene_plate_df = pd.DataFrame(gene_plate_stats)
        gene_plate_df.to_csv(output_folder / f"gene_plate_stats_{feature}.csv", index=False)
        
        gene_consistency = []
        for gene in genes:
            gene_data = gene_plate_df[gene_plate_df['gene'] == gene]
            if len(gene_data) >= 2:
                plate_means = gene_data['mean'].values
                gene_consistency.append({
                    'gene': gene,
                    'n_plates': len(gene_data),
                    'mean_across_plates': np.mean(plate_means),
                    'sd_across_plates': np.std(plate_means, ddof=1),
                    'cv_across_plates': (np.std(plate_means, ddof=1) / np.mean(plate_means) * 100) if np.mean(plate_means) != 0 else np.nan,
                    'range': np.max(plate_means) - np.min(plate_means)
                })
        
        consistency_df = pd.DataFrame(gene_consistency)
        consistency_df = consistency_df.sort_values('cv_across_plates')
        consistency_df.to_csv(output_folder / f"gene_consistency_{feature}.csv", index=False)
        
        # Generate CV plot (NEW)
        self._generate_cv_plot(consistency_df, feature, output_folder)
        
        self._write_gene_consistency_summary(consistency_df, feature, output_folder)
    
    def _generate_cv_plot(self, consistency_df, feature, output_folder):
        """Generate CV plot for all genes"""
        consistency_df = consistency_df.sort_values('gene')
        
        fig, ax = plt.subplots(figsize=(14, 6), dpi=DPI)
        
        genes = consistency_df['gene'].values
        cvs = consistency_df['cv_across_plates'].values
        
        x_pos = np.arange(len(genes))
        ax.bar(x_pos, cvs, color='steelblue', alpha=0.8, edgecolor='black', linewidth=0.5)
        
        ax.set_xlabel('Gene', fontsize=12, fontweight='bold')
        ax.set_ylabel('CV across plates (%)', fontsize=12, fontweight='bold')
        ax.set_title(f'Gene Variability Across Plates: {feature}', fontsize=14, fontweight='bold')
        ax.set_xticks(x_pos)
        ax.set_xticklabels(genes, rotation=90, ha='right', fontsize=8)
        ax.grid(True, alpha=0.3, axis='y', linestyle='--')
        
        plt.tight_layout()
        plt.savefig(output_folder / f"gene_cv_plot_{feature}.png", dpi=DPI, bbox_inches='tight')
        plt.close()
    
    def _write_gene_consistency_summary(self, consistency_df, feature, output_folder):
        """Write summary of gene consistency across plates"""
        unit = FEATURE_UNITS.get(feature, '')
        
        with open(output_folder / f"gene_consistency_summary_{feature}.txt", 'w') as f:
            f.write("="*80 + "\n")
            f.write(f"GENE CONSISTENCY ACROSS PLATES: {feature}\n")
            f.write("="*80 + "\n\n")
            
            f.write("Most Consistent Genes (lowest CV across plates):\n")
            f.write("-"*80 + "\n")
            for _, row in consistency_df.head(10).iterrows():
                f.write(f"\n{row['gene']}:\n")
                f.write(f"  Mean Â± SD: {row['mean_across_plates']:.3f} Â± {row['sd_across_plates']:.3f} {unit}\n")
                f.write(f"  CV: {row['cv_across_plates']:.2f}%\n")
                f.write(f"  Range: {row['range']:.3f} {unit}\n")
                f.write(f"  Present in {int(row['n_plates'])} plates\n")
            
            f.write("\n" + "="*80 + "\n")
            f.write("Least Consistent Genes (highest CV across plates):\n")
            f.write("-"*80 + "\n")
            for _, row in consistency_df.tail(10).iterrows():
                f.write(f"\n{row['gene']}:\n")
                f.write(f"  Mean Â± SD: {row['mean_across_plates']:.3f} Â± {row['sd_across_plates']:.3f} {unit}\n")
                f.write(f"  CV: {row['cv_across_plates']:.2f}%\n")
                f.write(f"  Range: {row['range']:.3f} {unit}\n")
                f.write(f"  Present in {int(row['n_plates'])} plates\n")


# ============================================================================
# ANALYSIS 3: TOTAL CELL COUNT PER PLATE
# ============================================================================

class PlateCellCountAnalyzer:
    """Analyzes total cell count per plate (technical QC)."""
    
    def __init__(self, df, output_base):
        self.df = df
        self.output_base = Path(output_base)
    
    def analyze(self):
        """Run cell count analysis"""
        print("\n" + "="*80)
        print("ANALYSIS 3: TOTAL CELL COUNT PER PLATE")
        print("="*80)
        
        output_folder = self.output_base / "03_plate_cell_counts"
        output_folder.mkdir(parents=True, exist_ok=True)
        
        cell_counts = self.df.groupby('plate').size().reset_index(name='total_cells')
        cell_counts = cell_counts.sort_values('plate')
        cell_counts.to_csv(output_folder / "plate_cell_counts.csv", index=False)
        
        fig, ax = plt.subplots(figsize=(10, 6), dpi=DPI)
        plates = cell_counts['plate'].values
        counts = cell_counts['total_cells'].values
        
        ax.bar(range(len(plates)), counts, color='steelblue', alpha=0.8, edgecolor='black')
        ax.set_xlabel('Plate', fontsize=12, fontweight='bold')
        ax.set_ylabel('Total Number of Cells', fontsize=12, fontweight='bold')
        ax.set_title('Total Cell Count Per Plate', fontsize=14, fontweight='bold')
        ax.set_xticks(range(len(plates)))
        ax.set_xticklabels(plates)
        ax.grid(True, alpha=0.3, axis='y', linestyle='--')
        
        mean_count = np.mean(counts)
        ax.axhline(y=mean_count, color='red', linestyle='--', linewidth=2, 
                  label=f'Mean: {mean_count:,.0f}')
        ax.legend()
        
        for i, (plate, count) in enumerate(zip(plates, counts)):
            ax.text(i, count, f'{count:,}', ha='center', va='bottom', fontweight='bold')
        
        plt.tight_layout()
        plt.savefig(output_folder / "plate_cell_counts.png", dpi=DPI, bbox_inches='tight')
        plt.close()
        
        with open(output_folder / "plate_cell_counts_summary.txt", 'w') as f:
            f.write("="*80 + "\n")
            f.write("TOTAL CELL COUNT PER PLATE\n")
            f.write("="*80 + "\n\n")
            for _, row in cell_counts.iterrows():
                f.write(f"{row['plate']}: {row['total_cells']:,} cells\n")
            f.write("\n" + "-"*80 + "\n")
            f.write(f"Mean: {np.mean(counts):,.0f} cells\n")
            f.write(f"SD: {np.std(counts, ddof=1):,.0f} cells\n")
            f.write(f"CV: {(np.std(counts, ddof=1) / np.mean(counts) * 100):.2f}%\n")
            f.write(f"Range: {np.min(counts):,} - {np.max(counts):,} cells\n")
        
        print("\n  [OK] Cell count analysis complete")


# ============================================================================
# ANALYSIS 4: GLOBAL WT VS GENE COMPARISON (ALL PLATES AGGREGATED)
# ============================================================================

class GlobalWTComparisonAnalyzer:
    """Compare each gene to WT using ALL cells from ALL plates."""
    
    def __init__(self, df, output_base):
        self.df = df
        self.output_base = Path(output_base)
    
    def analyze_all_features(self):
        """Run analysis for all features"""
        print("\n" + "="*80)
        print("ANALYSIS 4: GLOBAL WT VS GENE COMPARISON (ALL PLATES)")
        print("="*80)
        
        output_folder = self.output_base / "04_global_wt_comparisons"
        output_folder.mkdir(parents=True, exist_ok=True)
        
        for feature in MORPHOLOGY_FEATURES:
            print(f"\n  Processing: {feature}")
            self._analyze_feature(feature, output_folder)
        
        print("\n  [OK] Global WT comparison analysis complete")
    
    def _analyze_feature(self, feature, output_folder):
        """Analyze gene vs WT for a single feature across all plates"""
        wt_df = self.df[self.df['gene'] == 'WT']
        wt_values = wt_df[feature].values.astype(np.float32)
        wt_mean = np.mean(wt_values)
        wt_sd = np.std(wt_values, ddof=1)
        
        genes = [g for g in self.df['gene'].unique() if g != 'WT']
        results = []
        
        for gene in tqdm(genes, desc="  Genes", leave=False):
            gene_df = self.df[self.df['gene'] == gene]
            gene_values = gene_df[feature].values.astype(np.float32)
            
            if len(gene_values) < 30:
                continue
            
            gene_mean = np.mean(gene_values)
            gene_sd = np.std(gene_values, ddof=1)
            cohens_d = EffectSizeCalculator.cohens_d(gene_values, wt_values)
            
            results.append({
                'gene': gene,
                'n_cells': len(gene_values),
                'mean': gene_mean,
                'sd': gene_sd,
                'cohens_d': cohens_d,
                'interpretation': EffectSizeCalculator.interpret_cohens_d(cohens_d)
            })
            
            self._generate_comparison_plot(gene, gene_values, wt_values, 
                                          cohens_d, feature, output_folder)
        
        effect_df = pd.DataFrame(results)
        effect_df = effect_df.sort_values('cohens_d', key=abs, ascending=False)
        effect_df.to_csv(output_folder / f"global_gene_effect_sizes_{feature}.csv", index=False)
        
        self._write_global_summary(effect_df, wt_mean, wt_sd, len(wt_values), 
                                   feature, output_folder)
        self._generate_grouped_matrix_plot(feature, wt_values, output_folder)
    
    def _generate_comparison_plot(self, gene, gene_values, wt_values, cohens_d, 
                                  feature, output_folder):
        """Generate WT comparison density plot for a single gene"""
        fig, ax = plt.subplots(figsize=FIGURE_SIZE, dpi=DPI)
        
        x_min = np.percentile(self.df[feature], 0.1)
        x_max = np.percentile(self.df[feature], 99.9)
        bin_width = BIN_WIDTHS.get(feature, 0.1)
        bins = np.arange(x_min, x_max + bin_width, bin_width)
        
        ax.hist(wt_values, bins=bins, alpha=0.5, density=True, 
               label='WT (all plates)', color='gray', edgecolor='black', linewidth=0.5)
        ax.hist(gene_values, bins=bins, alpha=0.5, density=True, 
               label=gene, color='red', edgecolor='black', linewidth=0.5)
        
        interpretation = EffectSizeCalculator.interpret_cohens_d(cohens_d)
        unit = FEATURE_UNITS.get(feature, '')
        ax.set_xlabel(f'{feature} {unit}'.strip(), fontsize=12)
        ax.set_ylabel('Density', fontsize=12)
        ax.set_title(f'{gene} vs WT (all plates) | Cohen\'s d = {cohens_d:.2f} ({interpretation})', 
                    fontsize=12, weight='bold')
        ax.legend(loc='best', fontsize=10)
        ax.grid(True, alpha=0.3, linestyle='--')
        
        plt.tight_layout()
        plt.savefig(output_folder / f"{gene}_vs_WT_{feature}.png", 
                   dpi=DPI, bbox_inches='tight')
        plt.close()
    
    def _write_global_summary(self, effect_df, wt_mean, wt_sd, wt_n, 
                             feature, output_folder):
        """Write summary of global comparisons"""
        unit = FEATURE_UNITS.get(feature, '')
        
        with open(output_folder / f"global_summary_{feature}.txt", 'w') as f:
            f.write("="*80 + "\n")
            f.write(f"GLOBAL WT VS GENE COMPARISON: {feature}\n")
            f.write(f"(All cells from all plates aggregated)\n")
            f.write("="*80 + "\n\n")
            
            f.write(f"WT REFERENCE (ALL PLATES):\n")
            f.write(f"  {feature}: {wt_mean:.3f} Â± {wt_sd:.3f} {unit}\n")
            f.write(f"  (n={wt_n:,} cells)\n\n")
            
            f.write("="*80 + "\n")
            f.write("GENE COMPARISONS TO WT:\n")
            f.write("="*80 + "\n\n")
            
            for _, row in effect_df.iterrows():
                f.write(f"\n{'-'*80}\n")
                f.write(f"Gene: {row['gene']}\n")
                f.write(f"{'-'*80}\n")
                f.write(f"  {feature}: {row['mean']:.3f} Â± {row['sd']:.3f} {unit}\n")
                f.write(f"  (n={row['n_cells']:,} cells)\n")
                f.write(f"  Cohen's d vs WT: {row['cohens_d']:.3f} ({row['interpretation']})\n")
    
    def _generate_grouped_matrix_plot(self, feature, wt_values, output_folder):
        """Generate matrix plot with grouped genes (4 columns)"""
        all_genes = [g for g in self.df['gene'].unique() if g != 'WT']
        grouped_genes = sorted(set(get_grouped_gene_name(g) for g in all_genes))
        
        if len(grouped_genes) == 0:
            return
        
        grouped_effect_sizes = []
        grouped_values_list = []
        
        for grouped_gene in grouped_genes:
            mask = self.df['gene'].apply(lambda x: get_grouped_gene_name(x) == grouped_gene)
            grouped_df = self.df[mask]
            grouped_values = grouped_df[feature].values.astype(np.float32)
            effect_size = EffectSizeCalculator.cohens_d(grouped_values, wt_values)
            grouped_effect_sizes.append(effect_size)
            grouped_values_list.append(grouped_values)
        
        n_genes = len(grouped_genes)
        n_cols = 4
        n_rows = (n_genes + n_cols - 1) // n_cols
        
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 4 * n_rows), dpi=DPI)
        if n_rows == 1:
            axes = axes.reshape(1, -1)
        
        x_min = np.percentile(self.df[feature], 0.1)
        x_max = np.percentile(self.df[feature], 99.9)
        bin_width = BIN_WIDTHS.get(feature, 0.1)
        bins = np.arange(x_min, x_max + bin_width, bin_width)
        
        for idx, (grouped_gene, effect_size, gene_values) in enumerate(
            zip(grouped_genes, grouped_effect_sizes, grouped_values_list)):
            row = idx // n_cols
            col = idx % n_cols
            ax = axes[row, col]
            
            ax.hist(wt_values, bins=bins, alpha=0.5, density=True, label='WT', 
                   color='gray', edgecolor='black', linewidth=0.3)
            ax.hist(gene_values, bins=bins, alpha=0.5, density=True, label=grouped_gene, 
                   color='red', edgecolor='black', linewidth=0.3)
            
            interpretation = EffectSizeCalculator.interpret_cohens_d(effect_size)
            unit = FEATURE_UNITS.get(feature, '')
            ax.set_xlabel(f'{feature} {unit}'.strip(), fontsize=9)
            ax.set_ylabel('Density', fontsize=9)
            ax.set_title(f'{grouped_gene} | d={effect_size:.2f} ({interpretation})', 
                        fontsize=10, weight='bold')
            ax.legend(loc='best', fontsize=8)
            ax.grid(True, alpha=0.2, linestyle='--')
        
        for idx in range(n_genes, n_rows * n_cols):
            row = idx // n_cols
            col = idx % n_cols
            axes[row, col].axis('off')
        
        plt.tight_layout()
        plt.savefig(output_folder / f"grouped_genes_matrix_{feature}.png", 
                   dpi=DPI, bbox_inches='tight')
        plt.close()


# ============================================================================
# MAIN PIPELINE
# ============================================================================

def main():
    """Main execution pipeline for multi-plate analysis."""
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    output_base = Path(OUTPUT_BASE_FOLDER) / f"MultiPlate_Analysis_{timestamp}"
    output_base.mkdir(parents=True, exist_ok=True)
    
    print("="*80)
    print("MULTI-PLATE MORPHOLOGY ANALYSIS PIPELINE")
    print("="*80)
    print(f"Number of plates: {len(PLATE_PATHS)}")
    print(f"Output: {output_base}")
    print("="*80)
    
    df = load_all_plates(max_workers=6)
    
    wt_variability_analyzer = WTPlateVariabilityAnalyzer(df, output_base)
    wt_variability_analyzer.analyze_all_features()
    
    gene_variability_analyzer = GenePlateVariabilityAnalyzer(df, output_base)
    gene_variability_analyzer.analyze_all_features()
    
    cell_count_analyzer = PlateCellCountAnalyzer(df, output_base)
    cell_count_analyzer.analyze()
    
    global_wt_analyzer = GlobalWTComparisonAnalyzer(df, output_base)
    global_wt_analyzer.analyze_all_features()
    
    print("\n" + "="*80)
    print("MULTI-PLATE ANALYSIS COMPLETE")
    print("="*80)
    print(f"Results saved to: {output_base}")
    print("\nAnalysis folders:")
    print("  01_wt_plate_variability/ - WT variability across plates (PRIMARY)")
    print("  02_gene_plate_variability/ - Gene consistency across plates")
    print("  03_plate_cell_counts/ - Total cells per plate (QC)")
    print("  04_global_wt_comparisons/ - Gene vs WT (all plates aggregated)")


if __name__ == "__main__":
    main()